Import libraries and load TED Talks dataset

We use the cleaned dataset from your previous tasks. If it contains an ID column, we reuse it. Otherwise, we generate clearly labelled sample IDs.

In [ ]:
import pandas as pd
import numpy as np
import os

from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity

# Load cleaned TED Talks dataset
df = pd.read_csv("/content/cleaned_ted_talks.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# Identify actual column names
columns = {c.lower().strip(): c for c in df.columns}

# Find title column
if "title" not in columns:
    raise ValueError("Title column not found!")

title_column = columns["title"]

# Check whether an existing TED Talk ID is available
id_column = None

for name in ["ted_talk_id", "talk_id", "id", "talkid"]:
    if name in columns:
        id_column = columns[name]
        break

# Select six TED Talks
if id_column:
    talks = df[[id_column, title_column]].dropna().copy()
    talks = talks.drop_duplicates(subset=[id_column])
    talks = talks.head(6)

    talks.columns = ["ted_talk_id", "talk_title"]

    talks["ted_talk_id"] = talks["ted_talk_id"].astype(str)

    id_note = "Existing IDs from the cleaned dataset."

else:
    talks = df[[title_column]].dropna().head(6).copy()

    talks.columns = ["talk_title"]

    talks["ted_talk_id"] = [
        "TED_SAMPLE_" + str(i).zfill(3)
        for i in range(1, len(talks) + 1)
    ]

    id_note = "Generated sample IDs; these are not official TED IDs."

if len(talks) < 6:
    raise ValueError("At least six TED Talks are required.")

talks = talks.reset_index(drop=True)

talk_ids = talks["ted_talk_id"].tolist()

# Create ID-to-title mapping
talk_title_map = dict(
    zip(talks["ted_talk_id"], talks["talk_title"])
)

print("\nSelected TED Talks:")
print(id_note)

display(talks)

Dataset loaded successfully!
Dataset shape: (2550, 18)

Selected TED Talks:
Generated sample IDs; these are not official TED IDs.


,talk_title,ted_talk_id
0,Do schools kill creativity?,TED_SAMPLE_001
1,Averting the climate crisis,TED_SAMPLE_002
2,Simplicity sells,TED_SAMPLE_003
3,Greening the ghetto,TED_SAMPLE_004
4,The best stats you've ever seen,TED_SAMPLE_005
5,Why we do what we do,TED_SAMPLE_006


Create simulated user interaction dataset
We'll create five users with different rating patterns.

An important part of this example is that U001 and U002 have similar rating patterns. U002 likes a TED Talk that U001 has not interacted with. This will help us demonstrate collaborative filtering later.

In [ ]:
# Simulated user ratings
# Dictionary format: User ID -> Talk position -> Rating

ratings = {
    "U001": {0: 5, 1: 4, 2: 5, 3: 2, 4: 1},

    "U002": {0: 5, 1: 4, 2: 4, 3: 2, 4: 1, 5: 5},

    "U003": {0: 1, 1: 2, 2: 1, 3: 5, 4: 5, 5: 1},

    "U004": {0: 4, 1: 5, 2: 4, 3: 1, 4: 2, 5: None},

    "U005": {0: 2, 1: 1, 2: 2, 3: 5, 4: 4, 5: 1}
}

# Create interaction records
interaction_data = []

for user_id, user_ratings in ratings.items():

    for position, rating in user_ratings.items():

        talk_id = talk_ids[position]

        # Generate simulated feedback
        if rating is None:
            liked = True
            saved = False
        else:
            liked = rating >= 3
            saved = rating >= 4

        interaction_data.append({

            "user_id": user_id,

            "ted_talk_id": talk_id,

            "talk_title": talk_title_map[talk_id],

            "rating": rating,

            "liked": liked,

            "watched": True,

            "saved": saved
        })

# Convert into dataframe
interactions = pd.DataFrame(interaction_data)

# Check duplicate user-talk combinations
if interactions.duplicated(
    subset=["user_id", "ted_talk_id"]
).any():

    raise ValueError("Duplicate user-talk interactions found!")

print("SIMULATED INTERACTION DATA CREATED!")

print("Total records:", len(interactions))

display(interactions)

SIMULATED INTERACTION DATA CREATED!
Total records: 29


,user_id,ted_talk_id,talk_title,rating,liked,watched,saved
0,U001,TED_SAMPLE_001,Do schools kill creativity?,5.0,True,True,True
1,U001,TED_SAMPLE_002,Averting the climate crisis,4.0,True,True,True
2,U001,TED_SAMPLE_003,Simplicity sells,5.0,True,True,True
3,U001,TED_SAMPLE_004,Greening the ghetto,2.0,False,True,False
4,U001,TED_SAMPLE_005,The best stats you've ever seen,1.0,False,True,False
5,U002,TED_SAMPLE_001,Do schools kill creativity?,5.0,True,True,True
6,U002,TED_SAMPLE_002,Averting the climate crisis,4.0,True,True,True
7,U002,TED_SAMPLE_003,Simplicity sells,4.0,True,True,True
8,U002,TED_SAMPLE_004,Greening the ghetto,2.0,False,True,False
9,U002,TED_SAMPLE_005,The best stats you've ever seen,1.0,False,True,False


Build the user-item interaction matrix
We convert the interaction dataset into a matrix.

In [ ]:
# Create user-item matrix
user_item_matrix = interactions.pivot(
    index="user_id",
    columns="ted_talk_id",
    values="rating"
)

print("USER-ITEM INTERACTION MATRIX")

display(user_item_matrix)

print("\nMatrix Shape:", user_item_matrix.shape)

USER-ITEM INTERACTION MATRIX


ted_talk_id,TED_SAMPLE_001,TED_SAMPLE_002,TED_SAMPLE_003,TED_SAMPLE_004,TED_SAMPLE_005,TED_SAMPLE_006
user_id,,,,,,
U001,5.0,4.0,5.0,2.0,1.0,NaN
U002,5.0,4.0,4.0,2.0,1.0,5.0
U003,1.0,2.0,1.0,5.0,5.0,1.0
U004,4.0,5.0,4.0,1.0,2.0,NaN
U005,2.0,1.0,2.0,5.0,4.0,1.0



Matrix Shape: (5, 6)


Analyse user interaction data
This cell answers all five analysis requirements.
1. Number of unique users.
2. Number of unique TED Talks.
3. Talks with the highest ratings.
4. Users who rated the most talks.
5. Talks with the most interactions.

In [ ]:
# 1. Number of unique users
total_users = interactions["user_id"].nunique()

# 2. Number of unique TED Talks
total_talks = interactions["ted_talk_id"].nunique()

# Interaction count:
# One row represents one recorded user-talk interaction.
# It can contain rating, like/dislike, watch or save activity.

total_interactions = len(interactions)

print("TOTAL USERS:", total_users)
print("TOTAL TED TALKS:", total_talks)
print("TOTAL INTERACTIONS:", total_interactions)


# 3. Talks with the highest average ratings
talk_analysis = interactions.groupby("ted_talk_id").agg(

    Average_Rating=("rating", "mean"),

    Rating_Count=("rating", "count"),

    Total_Interactions=("ted_talk_id", "count")

).reset_index()

talk_analysis["Talk Title"] = talk_analysis[
    "ted_talk_id"
].map(talk_title_map)

# Find highest average rating
highest_average = talk_analysis["Average_Rating"].max()

highest_rated_talks = talk_analysis[
    talk_analysis["Average_Rating"] == highest_average
]

print("\nTALKS WITH HIGHEST AVERAGE RATINGS")

display(highest_rated_talks)


# 4. Users who rated the most talks
user_ratings = interactions.groupby("user_id")["rating"].count()

most_rated_count = user_ratings.max()

most_active_users = user_ratings[
    user_ratings == most_rated_count
]

print("\nUSERS WHO RATED THE MOST TALKS")

display(most_active_users)


# 5. Talks with the most interactions
most_interactions = talk_analysis["Total_Interactions"].max()

popular_talks = talk_analysis[
    talk_analysis["Total_Interactions"] == most_interactions
]

print("\nTALKS WITH THE MOST INTERACTIONS")

display(
    popular_talks[
        ["ted_talk_id", "Talk Title", "Total_Interactions"]
    ]
)

TOTAL USERS: 5
TOTAL TED TALKS: 6
TOTAL INTERACTIONS: 29

TALKS WITH HIGHEST AVERAGE RATINGS


,ted_talk_id,Average_Rating,Rating_Count,Total_Interactions,Talk Title
0,TED_SAMPLE_001,3.4,5,5,Do schools kill creativity?



USERS WHO RATED THE MOST TALKS


,rating
user_id,
U002,6
U003,6
U005,6



TALKS WITH THE MOST INTERACTIONS


,ted_talk_id,Talk Title,Total_Interactions
0,TED_SAMPLE_001,Do schools kill creativity?,5
1,TED_SAMPLE_002,Averting the climate crisis,5
2,TED_SAMPLE_003,Simplicity sells,5
3,TED_SAMPLE_004,Greening the ghetto,5
4,TED_SAMPLE_005,The best stats you've ever seen,5


Understand user preferences and collaborative filtering
First, we display each user's highly rated TED Talks.

Then we compare U001 and U002 using their common ratings.

In [ ]:
# Display user preferences based on high ratings
print("USER PREFERENCES")

for user_id in user_item_matrix.index:

    user_ratings = user_item_matrix.loc[user_id]

    preferred_talk_ids = user_ratings[
        user_ratings >= 4
    ].index.tolist()

    preferred_titles = [
        talk_title_map[talk_id]
        for talk_id in preferred_talk_ids
    ]

    print("\nUser:", user_id)

    print("Highly rated talks:")

    for title in preferred_titles:
        print("-", title)


# Collaborative filtering example
print("\nCOLLABORATIVE FILTERING EXAMPLE")

user1 = "U001"
user2 = "U002"

# Get ratings of both users
user1_ratings = user_item_matrix.loc[user1]

user2_ratings = user_item_matrix.loc[user2]

# Find talks rated by both users
common_talks = (
    user1_ratings.notna() &
    user2_ratings.notna()
)

# Extract common ratings
ratings1 = user1_ratings[common_talks].values.reshape(1, -1)

ratings2 = user2_ratings[common_talks].values.reshape(1, -1)

# Calculate user similarity
similarity = cosine_similarity(
    ratings1,
    ratings2
)[0][0]

print("\nUser 1:", user1)
print("User 2:", user2)

print("Common rated talks:", common_talks.sum())

print("User similarity:", round(similarity, 3))


# Check whether User 1 has interacted with Talk 6
target_talk_id = talk_ids[5]

user1_rating = user_item_matrix.loc[user1, target_talk_id]

user2_rating = user_item_matrix.loc[user2, target_talk_id]

print("\nUNSEEN TALK RECOMMENDATION")

print("Talk:", talk_title_map[target_talk_id])

if pd.isna(user1_rating) and user2_rating >= 4:

    print(user1, "has not rated this talk.")

    print(user2, "rated this talk:", user2_rating)

    print(
        "Recommendation:",
        "Suggest this talk to", user1,
        "because a similar user liked it."
    )

else:

    print("The example talk does not meet the recommendation condition.")

USER PREFERENCES

User: U001
Highly rated talks:
- Do schools kill creativity?
- Averting the climate crisis
- Simplicity sells

User: U002
Highly rated talks:
- Do schools kill creativity?
- Averting the climate crisis
- Simplicity sells
- Why we do what we do

User: U003
Highly rated talks:
- Greening the ghetto
- The best stats you've ever seen

User: U004
Highly rated talks:
- Do schools kill creativity?
- Averting the climate crisis
- Simplicity sells

User: U005
Highly rated talks:
- Greening the ghetto
- The best stats you've ever seen

COLLABORATIVE FILTERING EXAMPLE

User 1: U001
User 2: U002
Common rated talks: 5
User similarity: 0.995

UNSEEN TALK RECOMMENDATION
Talk: Why we do what we do
U001 has not rated this talk.
U002 rated this talk: 5.0
Recommendation: Suggest this talk to U001 because a similar user liked it.


Save CSV and generate observations PDF
This cell creates both required output files:
- data/interaction/user_interactions.csv
- reports/task_07_observations.pdf

In [ ]:
# Create required folders
os.makedirs("data/interaction", exist_ok=True)

os.makedirs("reports", exist_ok=True)


# Save interaction dataset
csv_path = "data/interaction/user_interactions.csv"

interactions.to_csv(csv_path, index=False)

print("Interaction CSV saved:", csv_path)


!pip install reportlab
# Generate observations PDF
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table
)
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from xml.sax.saxutils import escape

pdf_path = "reports/task_07_observations.pdf"

pdf = SimpleDocTemplate(
    pdf_path,
    pagesize=A4
)

styles = getSampleStyleSheet()

content = []


# Title
content.append(Paragraph(
    "Task 7 - User Preferences and Interaction Data",
    styles["Title"]
))

content.append(Spacer(1, 15))


# Dataset description
content.append(Paragraph(
    "1. Dataset Description",
    styles["Heading2"]
))

content.append(Paragraph(
    "This task uses SIMULATED DATA created for demonstrating "
    "user preferences and personalized TED Talk recommendations. "
    "The dataset contains five sample users and six TED Talks.",
    styles["BodyText"]
))

content.append(Spacer(1, 10))


# Dataset summary
content.append(Paragraph(
    "2. Dataset Summary",
    styles["Heading2"]
))

summary_table = [
    ["Description", "Value"],
    ["Unique Users", str(total_users)],
    ["Unique TED Talks", str(total_talks)],
    ["Recorded Interactions", str(total_interactions)],
    ["Rating Scale", "1 to 5"]
]

content.append(Table(summary_table, style=[
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
    ("PADDING", (0, 0), (-1, -1), 7)
]))

content.append(Spacer(1, 10))


# Matrix explanation
content.append(Paragraph(
    "3. User-Item Interaction Matrix",
    styles["Heading2"]
))

content.append(Paragraph(
    "Rows represent users, columns represent TED Talks, and "
    "cells contain numerical ratings. Missing values (NaN) "
    "indicate that no numerical rating was recorded. They are "
    "not treated as zero or as automatic dislikes.",
    styles["BodyText"]
))

content.append(Spacer(1, 10))


# Analysis findings
content.append(Paragraph(
    "4. Analysis Findings",
    styles["Heading2"]
))

highest_titles = ", ".join(
    talk_title_map[talk_id]
    for talk_id in highest_rated_talks["ted_talk_id"]
)

most_active_names = ", ".join(most_active_users.index.tolist())

popular_titles = ", ".join(
    talk_title_map[talk_id]
    for talk_id in popular_talks["ted_talk_id"]
)

findings = [
    ["Analysis", "Finding"],
    ["Highest average-rated talk(s)", highest_titles],
    ["Users who rated the most talks", most_active_names],
    ["Talk(s) with most interactions", popular_titles]
]

content.append(Table(findings, style=[
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
    ("PADDING", (0, 0), (-1, -1), 6)
]))

content.append(Spacer(1, 10))


# User preferences
content.append(Paragraph(
    "5. User Preferences",
    styles["Heading2"]
))

content.append(Paragraph(
    "User preferences are identified using highly rated talks. "
    "A rating of 4 or 5 is treated as a positive preference signal "
    "for this demonstration. These signals help identify content "
    "that a user may be interested in.",
    styles["BodyText"]
))

content.append(Spacer(1, 10))


# Collaborative filtering
content.append(Paragraph(
    "6. Collaborative Filtering",
    styles["Heading2"]
))

content.append(Paragraph(
    "Collaborative filtering identifies users with similar rating "
    "patterns. It can recommend a talk that a similar user liked "
    "but the target user has not interacted with.",
    styles["BodyText"]
))

content.append(Spacer(1, 10))


# Personalization
content.append(Paragraph(
    "7. Personalization Observations",
    styles["Heading2"]
))

content.append(Paragraph(
    "Ratings, likes, dislikes, watch history and saved talks "
    "provide different signals about user interests. Combining "
    "these signals can help a recommendation system suggest "
    "more relevant TED Talks to individual users.",
    styles["BodyText"]
))

content.append(Spacer(1, 10))


# Limitations
content.append(Paragraph(
    "8. Conclusion",
    styles["Heading2"]
))

content.append(Paragraph(
    "This task demonstrates how simulated user interaction data "
    "can be organized into a user-item matrix and analysed to "
    "understand user preferences. The collaborative filtering "
    "example prepares the foundation for building a personalized "
    "recommendation system in a future task. These results do not "
    "represent actual TED user behaviour.",
    styles["BodyText"]
))


# Generate PDF
pdf.build(content)

print("\nPDF generated successfully:", pdf_path)

Interaction CSV saved: data/interaction/user_interactions.csv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 26.6 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [reportlab]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.



PDF generated successfully: reports/task_07_observations.pdf
